# IQM Metric Test (Analysis Report)

Pilot → protocol selection → final run → paired statistical analysis for **odra** vs **simulator** ansatze on IQM Spark.

**Execution scripts** (run from project root):

```bash
python scripts/run_iqm_metric_test.py --phase pilot --depth 2
python scripts/select_iqm_metric_protocol.py --run-dir <pilot_run_dir>
python scripts/run_iqm_metric_test.py --phase final --depth 2 --shots <chosen> --repeats <chosen>
python scripts/analyze_iqm_metric_test.py --run-dir <final_run_dir>
```

Reusable logic lives in `src/qbanknote` (`evaluation`, `stats`, `classification`, `weights`, `iqm`).


## Methodology

1. **Statevector baseline:** For each CV fold and ansatz, load trained weights and evaluate accuracy/F1 on the test split using exact statevector simulation.
2. **Hardware inference:** Run the same checkpoints on IQM Spark with configurable shots and repeated forward passes per (fold, ansatz, shots).
3. **Pilot shot selection:** Compare consecutive shot budgets; freeze the smallest shot count whose max fold-wise accuracy/F1 change is below protocol thresholds.
4. **Final run:** Repeat hardware evaluation at the frozen shot/repeat budget across all folds.
5. **Paired tests:** Per fold, compute ODRA − simulator differences for IQM accuracy/F1 and simulator–hardware gaps; apply exact Wilcoxon signed-rank and sign tests across folds.


## 1. Select Run Directory


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

from qbanknote.paths import ensure_importable, find_project_root

ensure_importable()

from qbanknote.evaluation import read_csv_or_empty

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
OUTPUT_ROOT = PROJECT_ROOT / "evaluation_and_comparison/iqm_spark/iqm_metric_outputs"

# Set explicitly, or None to pick newest final/pilot run.
RUN_DIR = None  # e.g. OUTPUT_ROOT / "final" / "final_depth2_20250616_120000"

if RUN_DIR is None:
    candidates = []
    for phase in ("final", "pilot"):
        phase_dir = OUTPUT_ROOT / phase
        if phase_dir.is_dir():
            candidates.extend(sorted(phase_dir.iterdir(), key=lambda p: p.name))
    if not candidates:
        raise FileNotFoundError(f"No metric-test runs found under {OUTPUT_ROOT}")
    run_dir = candidates[-1]
else:
    run_dir = Path(RUN_DIR)

summary_df = read_csv_or_empty(run_dir / "summary_comparison.csv")
run_df = read_csv_or_empty(run_dir / "run_level_results.csv")
statevector_df = read_csv_or_empty(run_dir / "statevector_results.csv")
protocol_path = run_dir / "protocol_recommendation.json"
protocol = json.loads(protocol_path.read_text()) if protocol_path.exists() else {}

print(f"Run directory: {run_dir}")
print(f"Summary rows: {len(summary_df)} | Run-level rows: {len(run_df)}")
if protocol:
    print(f"Frozen protocol: shots={protocol.get('chosen_shot')} repeats={protocol.get('chosen_repeats')}")
summary_df.head()


## 2. Statevector vs IQM by Fold


In [ ]:
if summary_df.empty:
    print("No summary_comparison.csv loaded.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, metric in zip(axes, ["iqm_mean_accuracy", "iqm_mean_f1"]):
        pivot = summary_df.pivot_table(index="fold", columns="ansatz", values=metric)
        pivot.plot(kind="bar", ax=ax)
        ax.set_title(metric)
        ax.set_xlabel("fold")
    fig.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, metric in zip(axes, ["statevector_accuracy", "statevector_f1"]):
        pivot = summary_df.pivot_table(index="fold", columns="ansatz", values=metric)
        pivot.plot(kind="bar", ax=ax)
        ax.set_title(f"statevector {metric}")
        ax.set_xlabel("fold")
    fig.tight_layout()
    plt.show()


## 3. Shot Stability (Pilot Runs)


In [ ]:
shot_stability = read_csv_or_empty(run_dir / "shot_stability.csv")
shot_stability_agg = read_csv_or_empty(run_dir / "shot_stability_aggregate.csv")

if shot_stability_agg.empty:
    print("No shot stability aggregates found (expected for final runs with one shot budget).")
else:
    display(shot_stability_agg)

if not shot_stability.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    for metric in ("abs_change_accuracy", "abs_change_f1"):
        grouped = shot_stability.groupby(["previous_shot", "current_shot"])[metric].mean()
        grouped.plot(kind="bar", ax=ax, alpha=0.7, label=metric)
    ax.set_title("Mean absolute shot-to-shot metric change")
    ax.legend()
    plt.tight_layout()
    plt.show()


## 4. Repeat Variability & QPU / Wall Time


In [ ]:
if run_df.empty:
    print("No run_level_results.csv loaded.")
else:
    successful = run_df[run_df.get("status", "success") == "success"] if "status" in run_df.columns else run_df
    if not successful.empty and "repeat_index" in successful.columns:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        for ax, metric in zip(axes, ["accuracy", "f1"]):
            successful.boxplot(column=metric, by=["ansatz", "repeat_index"], ax=ax)
            ax.set_title(metric)
        fig.suptitle("Repeat variability")
        plt.tight_layout()
        plt.show()

    if "qpu_time_total" in run_df.columns:
        fig, ax = plt.subplots(figsize=(8, 4))
        run_df.boxplot(column="qpu_time_total", by="ansatz", ax=ax)
        ax.set_title("QPU time per hardware forward pass")
        plt.suptitle("")
        plt.tight_layout()
        plt.show()

    if "wall_time_forward_s" in run_df.columns:
        fig, ax = plt.subplots(figsize=(8, 4))
        run_df.boxplot(column="wall_time_forward_s", by="ansatz", ax=ax)
        ax.set_title("Wall time per hardware forward pass")
        plt.suptitle("")
        plt.tight_layout()
        plt.show()


## 5. Paired Statistical Analysis


In [ ]:
paired_tests = read_csv_or_empty(run_dir / "paired_tests.csv")
paired_diffs = read_csv_or_empty(run_dir / "paired_fold_differences.csv")
ansatz_summary = read_csv_or_empty(run_dir / "ansatz_level_summary.csv")

if paired_tests.empty:
    print("No paired_tests.csv found. Run: python scripts/analyze_iqm_metric_test.py --run-dir", run_dir)
else:
    display(paired_tests)

if not paired_diffs.empty:
  fig, axes = plt.subplots(1, 2, figsize=(10, 4))
  for ax, col in zip(axes, [
      "iqm_accuracy_diff_odra_minus_simulator",
      "iqm_f1_diff_odra_minus_simulator",
  ]):
      paired_diffs.boxplot(column=col, by="fold", ax=ax)
      ax.set_title(col)
  fig.suptitle("ODRA − simulator paired fold differences")
  plt.tight_layout()
  plt.show()

if not ansatz_summary.empty:
    display(ansatz_summary)
